"""
========================================================================
VADER ROBUSTNESS CHECK
========================================================================

Reviewer-defense purpose: confirm that the LLM > NYT sentiment deviation
finding is not an artifact of DistilBERT-SST2 specifically. We re-score
all generations with VADER (a lexicon+rule-based sentiment classifier with
completely different assumptions from a transformer model) and check
whether the direction of deviation holds.

VADER vs DistilBERT-SST2:
  VADER:           lexicon + rule-based, social-media optimized,
                   no contextual embedding, fast
  DistilBERT-SST2: transformer, movie-review-trained, contextual,
                   slow on long text (we chunk it)

Agreement in DIRECTION (LLM > NYT) is the bar. Magnitudes will differ;
that is expected and fine. We report direction-of-deviation correlation
between the two classifiers.

Outputs:
  data/results/vader_axis2_nyt.csv           NYT articles, VADER scores
  data/results/vader_axis2_gpt.csv           GPT articles, VADER scores
  data/results/vader_axis2_gemini.csv        Gemini articles, VADER scores
  data/results/results_vader_robustness.csv  cross-classifier comparison

This notebook is self-contained. Run anytime after Notebook 03 completes.
Estimated runtime: 1-2 minutes (VADER is fast).
"""

In [ ]:

# =============================================================================
# CELL 1: Setup and install
# =============================================================================

import subprocess
import sys

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    print("VADER already installed.")
except ImportError:
    print("Installing vaderSentiment...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vaderSentiment"])
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import os
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import wilcoxon, pearsonr

# Paths - relative to this notebook (notebooks/), same convention as Notebook 01-05
ROOT = Path("..").resolve()
PROCESSED = ROOT / "data" / "processed"
GENERATIONS = ROOT / "data" / "generations"
RESULTS = ROOT / "data" / "results"

print(f"ROOT: {ROOT}")

In [ ]:

# =============================================================================
# CELL 2: VADER scorer (chunked for fair comparison with DistilBERT)
# =============================================================================

vader = SentimentIntensityAnalyzer()


def vader_chunked_score(text, chunk_words=300):
    """
    Score article using VADER on word-based chunks, matching the chunking
    spirit of the DistilBERT pipeline. Returns mean compound score.

    VADER's compound score is in [-1, +1], comparable to DistilBERT's
    pos_prob - neg_prob output range.

    chunk_words=300 approximates DistilBERT's 450-token chunks (token≈0.75
    words on average). The exact value matters less than consistency.
    """
    if not isinstance(text, str) or not text.strip():
        return None
    words = text.split()
    if len(words) == 0:
        return None
    chunks = [
        " ".join(words[i:i + chunk_words])
        for i in range(0, len(words), chunk_words)
    ]
    scores = [vader.polarity_scores(c)["compound"] for c in chunks]
    return float(np.mean(scores)) if scores else None


# Quick test
print("\nVADER smoke test:")
print(f"  Positive text:  {vader.polarity_scores('This is a wonderful achievement.')['compound']:+.3f}")
print(f"  Negative text:  {vader.polarity_scores('This is a terrible disaster.')['compound']:+.3f}")
print(f"  Neutral text:   {vader.polarity_scores('The meeting was held on Tuesday.')['compound']:+.3f}")


In [ ]:

# =============================================================================
# CELL 3: Score NYT real articles with VADER
# =============================================================================

print("\n" + "=" * 60)
print("Scoring NYT real articles with VADER...")
print("=" * 60)

sampled = pd.read_csv(PROCESSED / "experiment_sample.csv")
nyt_rows = []
for _, r in sampled.iterrows():
    score = vader_chunked_score(r["full_text"])
    nyt_rows.append({
        "sample_id": r["sample_id"],
        "news_desk": r["news_desk"],
        "model": "NYT (real)",
        "prompt_version": "NYT",
        "rep": -1,
        "vader_compound": score,
    })
vader_nyt = pd.DataFrame(nyt_rows)
vader_nyt.to_csv(RESULTS / "vader_axis2_nyt.csv", index=False)

print(f"Saved: vader_axis2_nyt.csv ({len(vader_nyt)} rows)")
print("\nNYT VADER scores by desk:")
print(vader_nyt.groupby("news_desk")["vader_compound"].agg(["count", "mean", "std"]).round(4))


In [ ]:

# =============================================================================
# CELL 4: Score GPT generations with VADER
# =============================================================================

print("\n" + "=" * 60)
print("Scoring GPT-4o-mini generations with VADER...")
print("=" * 60)

df_gpt = pd.read_csv(GENERATIONS / "generations_gpt.csv")
ok_gpt = df_gpt[~df_gpt["generated_full_text"].astype(str).str.startswith("[ERROR")]

gpt_rows = []
for _, r in ok_gpt.iterrows():
    score = vader_chunked_score(r["generated_full_text"])
    gpt_rows.append({
        "sample_id": r["sample_id"],
        "rep": r["rep"],
        "prompt_version": r["prompt_version"],
        "news_desk": r["news_desk"],
        "model": "GPT-4o-mini",
        "vader_compound": score,
    })
vader_gpt = pd.DataFrame(gpt_rows)
vader_gpt.to_csv(RESULTS / "vader_axis2_gpt.csv", index=False)
print(f"Saved: vader_axis2_gpt.csv ({len(vader_gpt)} rows)")

In [ ]:

# =============================================================================
# CELL 5: Score Gemini generations with VADER
# =============================================================================

print("\n" + "=" * 60)
print("Scoring Gemini-2.5-Flash generations with VADER...")
print("=" * 60)

df_gem = pd.read_csv(GENERATIONS / "generations_gemini.csv")
ok_gem = df_gem[~df_gem["generated_full_text"].astype(str).str.startswith("[ERROR")]

gem_rows = []
for _, r in ok_gem.iterrows():
    score = vader_chunked_score(r["generated_full_text"])
    gem_rows.append({
        "sample_id": r["sample_id"],
        "rep": r["rep"],
        "prompt_version": r["prompt_version"],
        "news_desk": r["news_desk"],
        "model": "Gemini-2.5-Flash",
        "vader_compound": score,
    })
vader_gem = pd.DataFrame(gem_rows)
vader_gem.to_csv(RESULTS / "vader_axis2_gemini.csv", index=False)
print(f"Saved: vader_axis2_gemini.csv ({len(vader_gem)} rows)")

In [ ]:

# =============================================================================
# CELL 5b: Score Claude generations with VADER (if present)
# =============================================================================

c_path = GENERATIONS / "generations_claude.csv"
if c_path.exists():
    print("\n" + "=" * 60)
    print("Scoring Claude-Haiku-4.5 generations with VADER...")
    print("=" * 60)
    df_cl = pd.read_csv(c_path)
    ok_cl = df_cl[~df_cl["generated_full_text"].astype(str).str.startswith("[ERROR")]
    cl_rows = []
    for _, r in ok_cl.iterrows():
        cl_rows.append({
            "sample_id": r["sample_id"], "rep": r["rep"],
            "prompt_version": r["prompt_version"], "news_desk": r["news_desk"],
            "model": "Claude-Haiku-4.5",
            "vader_compound": vader_chunked_score(r["generated_full_text"]),
        })
    vader_claude = pd.DataFrame(cl_rows)
    vader_claude.to_csv(RESULTS / "vader_axis2_claude.csv", index=False)
    print(f"Saved: vader_axis2_claude.csv ({len(vader_claude)} rows)")
else:
    vader_claude = None
    print("generations_claude.csv not found - skipping Claude in the VADER check.")

In [ ]:

# =============================================================================
# CELL 6: Direction-only comparison: VADER vs DistilBERT
# =============================================================================

print("\n" + "=" * 78)
print("VADER ROBUSTNESS CHECK - direction-of-deviation comparison")
print("=" * 78)
print()
print("Question: Does VADER show the same direction of LLM-NYT deviation?")
print("(magnitude differences are expected; direction is what matters)")
print()


def agg_vader(df, model_label):
    agg = df.groupby(["sample_id", "prompt_version", "news_desk"],
                     as_index=False)["vader_compound"].mean()
    agg["model"] = model_label
    return agg


vader_nyt_per = vader_nyt[["sample_id", "vader_compound"]].rename(
    columns={"vader_compound": "nyt_vader"})

parts = [
    agg_vader(vader_gpt, "GPT-4o-mini").merge(vader_nyt_per, on="sample_id"),
    agg_vader(vader_gem, "Gemini-2.5-Flash").merge(vader_nyt_per, on="sample_id"),
]
if vader_claude is not None:
    parts.append(agg_vader(vader_claude, "Claude-Haiku-4.5").merge(vader_nyt_per, on="sample_id"))

vader_dev = pd.concat(parts, ignore_index=True)
vader_dev["vader_deviation"] = vader_dev["vader_compound"] - vader_dev["nyt_vader"]

print("VADER per-headline deviation (LLM - NYT) by model x prompt x desk:")
print()
vader_summary = vader_dev.groupby(["model", "prompt_version", "news_desk"]).agg(
    n=("vader_deviation", "count"),
    vader_dev_mean=("vader_deviation", "mean"),
    vader_dev_std=("vader_deviation", "std"),
).round(4).reset_index()
print(vader_summary.to_string(index=False))
vader_summary.to_csv(RESULTS / "results_vader_summary.csv", index=False)

In [ ]:

# =============================================================================
# CELL 7: Side-by-side DistilBERT vs VADER comparison table
# =============================================================================

# Load DistilBERT-based deviations (already computed in Notebook 02/03/04)
gpt_axis2 = pd.read_csv(RESULTS / "axis2_gpt.csv")
gem_axis2 = pd.read_csv(RESULTS / "axis2_gemini.csv")
nyt_axis2 = pd.read_csv(PROCESSED / "nyt_baseline_axis2.csv")


def agg_distil(df, model_label):
    agg = df.groupby(["sample_id", "prompt_version", "news_desk"],
                     as_index=False)["sentiment"].mean()
    agg["model"] = model_label
    return agg


distil_nyt_per = nyt_axis2[["sample_id", "sentiment"]].rename(
    columns={"sentiment": "nyt_distil"})

dparts = [
    agg_distil(gpt_axis2, "GPT-4o-mini").merge(distil_nyt_per, on="sample_id"),
    agg_distil(gem_axis2, "Gemini-2.5-Flash").merge(distil_nyt_per, on="sample_id"),
]
c_axis2 = RESULTS / "axis2_claude.csv"
if c_axis2.exists():
    dparts.append(agg_distil(pd.read_csv(c_axis2), "Claude-Haiku-4.5").merge(distil_nyt_per, on="sample_id"))

distil_dev = pd.concat(dparts, ignore_index=True)
distil_dev["distil_deviation"] = distil_dev["sentiment"] - distil_dev["nyt_distil"]

merged = vader_dev.merge(
    distil_dev[["sample_id", "model", "prompt_version", "news_desk",
                "distil_deviation", "sentiment", "nyt_distil"]],
    on=["sample_id", "model", "prompt_version", "news_desk"])

print("\n" + "=" * 78)
print("SIDE-BY-SIDE: DistilBERT-SST2 vs VADER mean deviation")
print("=" * 78)
print()
print(f"{'Model':<18s} {'Prompt':<6s} {'Desk':<10s}  "
      f"{'DistilBERT dev':>15s}  {'VADER dev':>10s}  {'Same dir?':>10s}")
print("-" * 78)

side_by_side = []
for _, group in merged.groupby(["model", "prompt_version", "news_desk"]):
    row = group.iloc[0]
    distil_mean = group["distil_deviation"].mean()
    vader_mean = group["vader_deviation"].mean()
    same_dir = "\u2713" if (distil_mean > 0) == (vader_mean > 0) else "\u2717"
    print(f"{row['model']:<18s} {row['prompt_version']:<6s} {row['news_desk']:<10s}  "
          f"{distil_mean:>+14.4f}  {vader_mean:>+9.4f}  {same_dir:>10s}")
    side_by_side.append({
        "model": row["model"], "prompt": row["prompt_version"], "desk": row["news_desk"],
        "distil_dev_mean": distil_mean, "vader_dev_mean": vader_mean,
        "agrees_in_direction": same_dir == "\u2713",
    })

side_df = pd.DataFrame(side_by_side)
n_total = len(side_df)
n_agree = int(side_df["agrees_in_direction"].sum())
print()
print(f"Direction agreement: {n_agree}/{n_total} cells ({100 * n_agree / n_total:.0f}%)")
side_df.to_csv(RESULTS / "results_vader_robustness.csv", index=False)
print(f"\nSaved: results_vader_robustness.csv")

In [ ]:
# =============================================================================
# CELL 8: Per-headline correlation between DistilBERT and VADER deviations
# =============================================================================

print("\n" + "=" * 78)
print("PER-HEADLINE CORRELATION: DistilBERT vs VADER deviation")
print("=" * 78)
print()
print("Within each (model, prompt) cell, do DistilBERT and VADER produce")
print("correlated per-headline deviations? High correlation = the two classifiers")
print("agree on which headlines show LLM > NYT.")
print()

corr_rows = []
for (model, prompt), group in merged.groupby(["model", "prompt_version"]):
    if len(group) < 5:
        continue
    r, p = pearsonr(group["distil_deviation"], group["vader_deviation"])
    print(f"  {model:<18s} {prompt}  n={len(group):2d}  "
          f"r = {r:+.3f}  p = {p:.4f}")
    corr_rows.append({
        "model": model, "prompt": prompt,
        "n": len(group), "r": r, "p": p,
    })

pd.DataFrame(corr_rows).to_csv(RESULTS / "results_vader_correlation.csv", index=False)


In [ ]:

# =============================================================================
# CELL 9: Paired Wilcoxon — does VADER also show LLM ≠ NYT?
# =============================================================================

print("\n" + "=" * 78)
print("VADER PAIRED WILCOXON — does VADER also reject LLM == NYT?")
print("=" * 78)
print()

vader_paired = []
for (model, prompt, desk), group in vader_dev.groupby(
    ["model", "prompt_version", "news_desk"]
):
    if len(group) < 5:
        continue
    try:
        stat, p = wilcoxon(
            group["vader_compound"].values,
            group["nyt_vader"].values,
            alternative="two-sided",
        )
    except Exception:
        stat, p = np.nan, np.nan
    sig = "✓" if p < 0.05 else " "
    print(f"  {model:<18s} {prompt} {desk:<10s}  n={len(group):2d}  "
          f"VADER LLM={group['vader_compound'].mean():+.4f}  "
          f"NYT={group['nyt_vader'].mean():+.4f}  "
          f"dev={group['vader_deviation'].mean():+.4f}  p={p:.4f}  {sig}")
    vader_paired.append({
        "model": model, "prompt": prompt, "desk": desk, "n": len(group),
        "vader_llm_mean": group["vader_compound"].mean(),
        "vader_nyt_mean": group["nyt_vader"].mean(),
        "vader_deviation": group["vader_deviation"].mean(),
        "wilcoxon_stat": stat, "p_value": p,
    })

pd.DataFrame(vader_paired).to_csv(RESULTS / "results_vader_paired.csv", index=False)

In [ ]:

# =============================================================================
# CELL 10: Summary for paper
# =============================================================================

print("\n" + "=" * 78)
print("SUMMARY FOR PAPER")
print("=" * 78)
print()

n_cells_total = len(side_df)
n_cells_agree = int(side_df["agrees_in_direction"].sum())
agreement_pct = 100 * n_cells_agree / n_cells_total
mean_corr = pd.DataFrame(corr_rows)["r"].mean()

vader_paired_df = pd.DataFrame(vader_paired)
n_significant_vader = int((vader_paired_df["p_value"] < 0.05).sum())

# DistilBERT significance: read from the 3-model cross-model paired results (A/E)
p3 = RESULTS / "results_crossmodel3_paired.csv"
if p3.exists():
    distil_paired = pd.read_csv(p3)
    n_significant_distil = int((distil_paired["p_value"] < 0.05).sum())
    n_distil_total = len(distil_paired)
else:
    n_significant_distil = n_distil_total = None

print(f"Direction agreement (DistilBERT vs VADER):  {n_cells_agree}/{n_cells_total} cells ({agreement_pct:.0f}%)")
print(f"Mean Pearson correlation (per-headline):     r = {mean_corr:+.3f}")
print(f"DistilBERT cells with p<0.05 vs NYT (A/E):   {n_significant_distil}/{n_distil_total}")
print(f"VADER cells with p<0.05 vs NYT:              {n_significant_vader}/{len(vader_paired_df)}")

print(f"""
Suggested paper text (for Methods or Discussion):

"As a robustness check on the DistilBERT-SST2 sentiment measure, we re-scored
all NYT and LLM-generated articles using VADER (Hutto & Gilbert 2014), a
lexicon- and rule-based sentiment classifier with substantially different
underlying assumptions from a transformer model. Across the {n_cells_total} (model,
prompt, desk) cells in our analysis, VADER and DistilBERT agreed on the
direction of LLM-NYT deviation in {n_cells_agree}/{n_cells_total} cells ({agreement_pct:.0f}%). Per-headline
deviations between the two classifiers were positively correlated (mean
Pearson r = {mean_corr:+.3f}). This direction-level convergence supports the
interpretation that the affective deviation we report is a real property of
the generated text, not an artifact of any single sentiment classifier."
""")

print()
print("Output files:")
for f in sorted(RESULTS.glob("*vader*")):
    print(f"  {f.name:45s}  {f.stat().st_size:>8,} bytes")